In [218]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [219]:
file_name = r"C:\Users\admin\Downloads\26.09.2023 £78.24 Heyner UK.pdf"

r"C:\Users\admin\Downloads\26.09.2023 £78.24 Heyner UK.pdf"

'C:\\Users\\admin\\Downloads\\26.09.2023 £78.24 Heyner UK.pdf'

In [220]:
invoice_type = "Products"

input_file = fr"C:\Users\admin\Downloads\11.01.2024 £26.40 Heyner UK.pdf"

In [221]:
table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(112,16,254,140),
                  columns=[140],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading = table1[0]
display(heading)

name = "Heyner UK"
docnum = heading[0][1]
print(docnum)

date = heading[0][5].replace(' ','/')
date = str(datetime.strptime(date, "%d/%b/%Y"))
print(date)

heading_length = len(heading)

if heading_length == 10:
    ordernum = heading[0][9]
else:
    ordernum = None

transfernum = None
print(ordernum)
print(transfernum)


,0
0,Invoice #
1,INV105
2,Order #
3,105
4,Invoice Date
5,11 Jan 2024
6,Payment Due
7,10 Feb 2024
8,P.O. #
9,PO32783 / MLP93286


INV105
2024-01-11 00:00:00
PO32783 / MLP93286
None


In [222]:
table2 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(285,38,833,570),
                  columns=[96,320,363,435,500,558],
                  pandas_options={'header': None},
                  encoding="windows-1254")
content=table2[0]

content

,0,1,2,3,4,5
0,347100,"HEYNER Premium Scissor Jack 1,0 ton",1.0,£19.00,20%,£19.00
1,NaN,Royal Mail 48 Tracked,1.0,£3.00,20%,£3.00
2,NaN,NaN,NaN,Sub-Total (ex. VA,T),£22.00
3,NaN,NaN,NaN,VAT Standard @ 2,0%,£4.40
4,NaN,NaN,NaN,Total (1 item),NaN,£26.40
5,hank you for,your buissnes.,NaN,NaN,NaN,NaN
6,ay Online,using Paypal,NaN,NaN,NaN,NaN
7,ther Payme,nt Options,NaN,NaN,NaN,NaN
8,EYNER UK L,TD,NaN,NaN,NaN,NaN
9,SBC 40-11-,65 account no. 40075930,NaN,NaN,NaN,NaN


In [223]:
#content = content[content[0].notnull()].iloc[:, 0:].reset_index(drop=True)  # Remove NaN
content = content.dropna(subset=[1,3]).reset_index(drop=True)  # Remove rows with NaN in column 1,3

display(content)

content[[3,5]] = content[[3,5]].replace('[£, ]','', regex=True).astype('float64')
content

,0,1,2,3,4,5
0,347100,"HEYNER Premium Scissor Jack 1,0 ton",1.0,£19.00,20%,£19.00
1,NaN,Royal Mail 48 Tracked,1.0,£3.00,20%,£3.00


,0,1,2,3,4,5
0,347100,"HEYNER Premium Scissor Jack 1,0 ton",1.0,19.0,20%,19.0
1,NaN,Royal Mail 48 Tracked,1.0,3.0,20%,3.0


In [224]:
content.rename(columns={
    0: 'Code',
    1: 'Item',
    2: 'Quantity',
    3: 'Unit Price',
    4: 'VAT Rate',
    5: 'Amount GBP'}, inplace=True)

display(content)

,Code,Item,Quantity,Unit Price,VAT Rate,Amount GBP
0,347100,"HEYNER Premium Scissor Jack 1,0 ton",1.0,19.0,20%,19.0
1,NaN,Royal Mail 48 Tracked,1.0,3.0,20%,3.0


In [225]:
dict_content = content.to_dict(orient='records')
dict_content

line_items=[]
for item in dict_content:
    # print(item)
    partNum = item['Code']
    desc = item['Item']
    quantity = item['Quantity']
    netTotal = item['Amount GBP']

    print(partNum)
    if isinstance(partNum, (float, np.float64)) and np.isnan(partNum):
        line_item = {"line_type":"shipping_expense",
                    "sku": None,
                    "name": desc,
                    "Quantity": int(quantity),
                    "net_total": float(netTotal),
                    "tax_type": "INPUT2"}
    else:
        line_item = {"line_type": "inventory",
                    "sku": partNum,
                    "name": desc,
                    "Quantity": int(quantity),
                    "net_total": float(netTotal),
                    "tax_type": "INPUT2"}      
    
    line_items.append(line_item)

print(line_items)

347100
nan
[{'line_type': 'inventory', 'sku': '347100', 'name': 'HEYNER Premium Scissor Jack 1,0 ton', 'Quantity': 1, 'net_total': 19.0, 'tax_type': 'INPUT2'}, {'line_type': 'Postage, Freight & Courier', 'sku': None, 'name': 'Royal Mail 48 Tracked', 'Quantity': 1, 'net_total': 3.0, 'tax_type': 'INPUT2'}]


In [226]:
table3 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(327,357,834,571),
                  columns=[480,571],
                  pandas_options={'header': None},
                  encoding='windows-1254')

total_content=table3[0]
display(total_content)

#To remove "£"
total_content[[1]] = total_content[[1]].replace('[£, ]','', regex=True).astype('float64')
display(total_content)


,0,1
0,£3.00 20%,£3.00
1,Sub-Total (ex. VAT),£22.00
2,VAT Standard @ 20%,£4.40
3,Total (1 item),£26.40


,0,1
0,£3.00 20%,3.0
1,Sub-Total (ex. VAT),22.0
2,VAT Standard @ 20%,4.4
3,Total (1 item),26.4


In [227]:
table3_length = len(total_content)
print(table3_length)

final_total = str(total_content[1][table3_length-1])
final_total = float(final_total.replace(","," "))
final_total

4


26.4

In [228]:
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        None,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Downloads\\26.09.2023 £78.24 Heyner UK.pdf',
 'Type': 'Products',
 'Name': 'Heyner UK',
 'Date': '2024-01-11 00:00:00',
 'Reference No.': 'INV105',
 'Order No.': 'PO32783 / MLP93286',
 'Transfer No.': None,
 'Document No.': None,
 'Line Items': [{'line_type': 'inventory',
   'sku': '347100',
   'name': 'HEYNER Premium Scissor Jack 1,0 ton',
   'Quantity': 1,
   'net_total': 19.0,
   'tax_type': 'INPUT2'},
  {'line_type': 'Postage, Freight & Courier',
   'sku': None,
   'name': 'Royal Mail 48 Tracked',
   'Quantity': 1,
   'net_total': 3.0,
   'tax_type': 'INPUT2'}],
 'Total': 26.4}